# Turning the Training/Test loops into functions


## Looping through batches

1. Loop through epochs
2. for each each epoch, loop through batches
3. on each batch, perform training steps, calculating the loss value *per batch*
4. also loop through test batches, with testing steps and a test loss per batch
5. Time it to understand the how to optimize the batch size




In [ ]:
import torch
print(torch.__version__)
from torch import nn, Tensor, optim

def train_step(
        model: nn.Module,
        X: Tensor,
        y: Tensor,
        loss_fn: nn.Module,
        optimizer: optim.Optimizer,
        is_classification: bool=False,
        eval_fn=None
    ) -> tuple[Tensor, Tensor|None]:
    """
    Execute a simple training step on the model based on the data
        Wheter the data is a batch or a full dataset
    It uses the loss function to guide the optimizer
    It `does not` set the model to train mode

    *** Parameters
    * `X`: The training data
    * `y`: The labels corresponding to the training data
    * `loss_fn`: Measures of predictions error compared with the labels `y`
        must receive the prediction and the labels in this specific order
    * `optimizer`: The optimizer used to calibrate the model parameters
        must be previously set to work with the model parameter
    * `is_classification`: Wheter is a classification mode or not
        If it is, is assumed that the `eval_fn` will require the use of `torch.argmax(dim=1)` to be applied on the predictions
    * `eval_fn`: Optional.
        A different function to measure the model current performance
        Must receive the parameters in the Scikit Learn order (that is, labels/y first, then the predictions)
        Must return a torch.Tensor-compatible value

    *** Returns
    * `loss`: the loss value
    * `eval`: the custom evaluation value, None if eval_fn is None
    """

    # Do the Forward Pass
    preds = model(X)

    # Calculate Loss
    loss = loss_fn(preds, y)

    # Optimizer Zero Grad
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Gradient Descent
    optimizer.step()

    eval = None
    if not eval_fn is None:
        p = preds.argmax(dim=1) if is_classification else preds
        eval = torch.tensor(eval_fn(y, p))

    return loss, eval


def test_step(
        model: nn.Module,
        X: Tensor,
        y: Tensor,
        loss_fn: nn.Module,
        is_classification: bool = False,
        eval_fn = None
    ) -> tuple[Tensor, Tensor|None]:
    """
    Execute a simple testing step on the model based on the data
        Wheter the data is a batch or a full dataset
    It `does not` set the model to eval mode nor inference mode

    *** Parameters
    * `X`: The testing data
    * `y`: The labels corresponding to the testing data
    * `loss_fn`: Measures of predictions error compared with the labels `y`
        must receive the prediction and the labels in this specific order
    * `is_classification`: Wheter is a classification mode or not
        If it is, is assumed that the `eval_fn` will require the use of `torch.argmax(dim=1)` to be applied on the predictions
    * `eval_fn`: Optional.
        A different function to measure the model current performance
        Must receive the parameters in the Scikit Learn order (that is, labels/y first, then the predictions)
        Must return a torch.Tensor-compatible value

    *** Returns
    * `loss`: the loss value
    * `eval`: the custom evaluation value, None if eval_fn is None
    """

    # Do the Forward Pass
    preds = model(X)

    # Calculate Loss
    loss = loss_fn(preds, y)

    eval = None
    if not eval_fn is None:
        p = preds.argmax(dim=1) if is_classification else preds
        eval = torch.tensor(eval_fn(y, p))

    return loss, eval


from torch.utils.data import DataLoader

def training_loop_batch(
        model: nn.Module,
        data: DataLoader,
        loss_fn: nn.Module,
        optimizer: optim.Optimizer,
        device: torch.device,
        eval_fn=None,
        is_classification: bool=False,
    ) -> tuple[Tensor, Tensor|None]:
    """
    Performs a epoch of model training
    Iterating through all the dataloader in batches

    *** Parameters
    * `data`: The dataloader contaning the data in batches
    * `loss_fn`: Measures of predictions error compared with the labels `y`
        must receive the prediction and the labels in this specific order
    * `optimizer`: The optimizer used to calibrate the model parameters
        must be previously set to work with the model parameter
    * `is_classification`: Wheter is a classification mode or not
        If it is, is assumed that the `eval_fn` will require the use of `torch.argmax(dim=1)` to be applied on the predictions
    * `eval_fn`: Optional.
        A different function to measure the model current performance
        Must receive the parameters in the Scikit Learn order (that is, labels/y first, then the predictions)
        Must return a torch.Tensor-compatible value

    *** Returns
    * `loss`: the loss value
    * `eval`: the custom evaluation value, None if eval_fn is None
    """

    # Sets the model to training mode
    model.train()
    accumulate_loss = torch.tensor(0).float().to(device)
    accumulate_eval = torch.tensor(0).float().to(device)
    size = torch.tensor(len(data)).float().to(device)

    for (X, y) in data:
        X = X.to(device)
        y = y.to(device)
        # performs a step
        loss, eval = train_step(
            model, X, y, loss_fn, optimizer,
            is_classification, eval_fn
        )
        accumulate_loss += loss

        if not eval is None:
            accumulate_eval += eval


    accumulate_loss /= size
    if eval_fn is None:
        accumulate_eval = None
    else:
        accumulate_eval /= size

    return accumulate_loss, accumulate_eval


def testing_loop_batch(
        model: nn.Module,
        data: DataLoader,
        loss_fn: nn.Module,
        device: torch.device,
        eval_fn=None,
        is_classification: bool=False,
    ) -> tuple[Tensor, Tensor|None]:
    """
    Performs a epoch of model testing
    Iterating through all the dataloader in batches

    *** Parameters
    * `data`: The dataloader contaning the data in batches
    * `loss_fn`: Measures of predictions error compared with the labels `y`
        must receive the prediction and the labels in this specific order
    * `is_classification`: Wheter is a classification mode or not
        If it is, is assumed that the `eval_fn` will require the use of `torch.argmax(dim=1)` to be applied on the predictions
    * `eval_fn`: Optional.
        A different function to measure the model current performance
        Must receive the parameters in the Scikit Learn order (that is, labels/y first, then the predictions)
        Must return a torch.Tensor-compatible value

    *** Returns
    * `loss`: the loss value
    * `eval`: the custom evaluation value, None if eval_fn is None
    """

    # Sets the model to training mode
    model.eval()

    accumulate_loss = torch.tensor(0).float().to(device)
    accumulate_eval = torch.tensor(0).float().to(device)
    size = torch.tensor(len(data)).float().to(device)

    with torch.inference_mode():
        for (X, y) in data:
            X = X.to(device)
            y = y.to(device)
            # performs a step
            loss, eval = test_step(
                model, X, y, loss_fn,
                is_classification, eval_fn
            )
            accumulate_loss += loss

            if not eval is None:
                accumulate_eval += eval

        accumulate_loss /= size
        if eval_fn is None:
            accumulate_eval = None
        else:
            accumulate_eval /= size

    return accumulate_loss, accumulate_eval


# progress bar
from tqdm.auto import tqdm # to automatically guess the environment
# To verify the speed performance
from timeit import default_timer as timer


def print_train_timer(start: float,
                      end: float,
                      device: torch.device = None
                      ) -> float:
    """
    prints the difference between start and end
    """
    total_time = end - start
    print(f"Total time on {device}: {total_time:.3f} seconds")
    return total_time


def eval_model(
        model: nn.Module,
        data_loader: DataLoader,
        loss_fn: nn.Module,
        device: torch.device,
        eval_fn=None,
        training_time:float=-1.0,
        ) -> dict:
    loss: Tensor = Tensor([0]).to(device)
    eval: Tensor = Tensor([0]).to(device)

    model.eval()
    with torch.inference_mode():
        for X, y in tqdm(data_loader):
            X = X.to(device)
            y = y.to(device)
            y_pred = model(X)

            loss += loss_fn(y_pred, y)
            if not eval_fn is None:
                eval += eval_fn(
                    y_true=y,
                    y_pred=y_pred.argmax(dim=1)
                )

        loss /= len(data_loader)
        eval /= len(data_loader)
    return {
        "model_name": model.__class__.__name__,
        "model_loss": loss.item(),
        "model_eval (%)": eval.item(),
        "device": str(device),
        "training_time (sec)": training_time,
        }


def train_model_with_batches(
        model: nn.Module,
        train_data: DataLoader,
        test_data: DataLoader,
        loss_fn: nn.Module,
        optimizer_fn,
        lr: float,
        N_EPOCHS: int,
        RND_SEED: int=None,
        eval_fn=None,
        is_classification: bool=False,
    ) -> dict:
    """
    Executes a full model training
    Iterating through all the dataloader in batches

    *** Parameters
    * `data`: The dataloader contaning the data in batches
    * `loss_fn`: Measures of predictions error compared with the labels `y`
        must receive the prediction and the labels in this specific order
    * `optimizer`: The optimizer module (non-instantiated) used to calibrate the model parameters
        Will be set to work with the model parameters
    * `lr`: Learning rate to be passed to the optimizer
    * `N_EPOCHS`: Number of training epochs
    * `RND_SEED`: Value for torch.manual_seed
    * `eval_fn`: Optional.
        A different function to measure the model current performance
        Must receive the parameters in the Scikit Learn order (that is, labels/y first, then the predictions)
        Must return a torch.Tensor-compatible value
    * `is_classification`: Wheter is a classification mode or not
        If it is, is assumed that the `eval_fn` will require the use of `torch.argmax(dim=1)` to be applied on the predictions
    """
    if not RND_SEED is None:
        torch.manual_seed(RND_SEED)

    optimizer = optimizer_fn(
        params=model.parameters(),
        lr=lr
    )
    device = next(model.parameters()).device

    # Start counting the time
    start = timer()

    for ep in range(1, N_EPOCHS+1):
        train_loss, _ = training_loop_batch(
            model,
            train_data,
            loss_fn,
            optimizer,
            device,
        )
        test_loss, test_eval = testing_loop_batch(
            model,
            train_data,
            loss_fn,
            device,
            eval_fn,
            is_classification,
        )
        print(f"Epoch: {ep}")
        print(f" - Train Loss: {train_loss:.4f}")
        print(f" - Test Loss:  {test_loss:.4f}")
        print(f" - Test Eval: {test_eval:.2f}%")

    training_time = print_train_timer(
        start,
        timer(),
        device,
    )

    return eval_model(
        model,
        test_dataloader,
        loss_fn,
        device,
        eval_fn,
        training_time,
    )

# Computer Vision

Using Machine Learning to recognize patterns in a image

Images are represented as:
* NCHW: [`batch_size` , `colour_channels` , `height` , `width`]
    * batch size: The number of images per training iteration
    * colour channels: The number of bits used to represent the colours
    * height x width: Image size
* Using colour_channels first is a pytorch-specifc decision
    * It's probable to be overturned eventually, as colour channels last is more efficient
* The most common batch size is 32

## Batch Size

It is very inefficient to iterate over thousands (possibly millions) of images to calculate the loss average of them all, as all of it would be stored in memory

Therefore, it's better to divide the dataset in batches, iterate through a batch, calculate the loss of it, actualize the model, then go to next batch
 - It not only is less computationally intensive, but allows for the NN to improve their gradient more times per epoch

## Convolutional Neural Networks (CNN)

Libraries:

 - `torchvision`: base dominion for COmputer Vision in PyTorch
 - `torchvision.datasets`: datasets and data loading functions
    - All datasets will be subclasses of `torch.utils.data.Dataset`
    - Hence, they can be loaded by `torch.utils.data.DataLoader`
    - In paralel using torch.multiprocessing
 - `torchvision.models`: Pretrained computer vision models to use
 - `torchvision.transforms`: functions for manipulation of images data to make it suitable to feeding ML models

# Loading Datasets

`torchvision` has a comprehensive variety of datasets
 - Like the **MNIST** (*Modified* National Institute of Standads and Technology) database
    - A large database of hand-written digits very usefull to train image processing systems
 - More specifically, **fashion-mnist**, a similar-style dataset of piece of clothing

In [ ]:
import torchvision as tv
print(tv.__version__)
# converts images to tensors C x H x W
# It receives a H x W x C image or ndarray
# if the ndarray is uint8 or the image mode right, it also scales it
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using Device:", device)

RND_SEED = 0

train_data = tv.datasets.FashionMNIST(
    root="data",            # where to download the data
    train=True,             # the training version
    download=True,          # to download
    transform= ToTensor(),  # the function to format the data
    target_transform=None   # the function to format the labels
)
test_data = tv.datasets.FashionMNIST(
    root="data",            # where to download the data
    train=False,            # the testing version
    download=True,          # to download
    transform= ToTensor(),  # the function to format the data
    target_transform=None   # the function to format the labels
)
print(train_data.class_to_idx)
print(train_data.targets)
class_names = train_data.classes
img, label = train_data[0]
COLORS, HEIGHT, WIDTH = img.shape
label_name = class_names[label]
print(f"Image Shape: {img.shape} -> [color_channels, height, width]")
print(f"Image Label: {label_name}")




fig = plt.figure(figsize=(9, 9))
rows, cols = 4, 4
torch.manual_seed(RND_SEED)
for i in range(1, rows*cols + 1):
    rnd_idx = int(torch.randint(0, len(train_data), size=[1]).item())
    img, label = train_data[rnd_idx]
    fig.add_subplot(rows, cols, i)
    # needs to be squeezed to respect imshow format
    plt.imshow(img.squeeze(), cmap="gray")
    plt.title(class_names[label])
    plt.axis(False)

# Setting the device to cuda would impede the dataloader from being iterated
# train_data.data = train_data.data.to(device)
# train_data.targets = train_data.targets.to(device)
# test_data.data = test_data.data.to(device)
# test_data.targets = test_data.targets.to(device)

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_dataloader = DataLoader(
    dataset=train_data,
    batch_size=BATCH_SIZE,
    shuffle=True # to make the batch random
)
test_dataloader = DataLoader(
    dataset=test_data,
    batch_size=BATCH_SIZE,
    shuffle=False # it's easier to test
)

print(f"Length of Train Dataloader: {len(train_dataloader)}, Batches of {BATCH_SIZE}")
print(f"Length of Test Dataloader: {len(test_dataloader)}, Batches of {BATCH_SIZE}")

In [ ]:
# a Flatten is a contiguous range of dims into a tensor
# that is, it turns a multidimentional tensor into a list
# allows for linear models to receive multi-dimentional data
flatten_model = nn.Flatten()

tmp = next(iter(train_dataloader))[0]

print(tmp.shape, "-> [colors, height, width]")
flat = flatten_model(tmp)
print(flat.shape, "-> [colors, height*width]")

# Building CV Models

It's best practice to start with a baseline model
 - a simple model to be improved through expermients

## Creating a Linear Model


In [ ]:
import torchmetrics as tm
from helper_functions import accuracy_fn

class FashionV0(nn.Module):
    def __init__(self,
                 input_shape: tuple[int, int],
                 hidden_units: int,
                 output_shape: int):
        super().__init__()

        height, width = input_shape

        self.layer_stack = nn.Sequential(
            nn.Flatten(), # first format the data
            nn.Linear(
                in_features=height*width,
                out_features=hidden_units
            ),
            nn.Linear(
                in_features=hidden_units,
                out_features=output_shape
            ),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layer_stack(x)

torch.manual_seed(RND_SEED)

model_0 = FashionV0(
    input_shape=(int(HEIGHT), int(WIDTH)),
    hidden_units=10,
    output_shape=len(class_names) # one for every class
).to(device)

print(model_0)
print("-----------------")

model_0_results = train_model_with_batches(
    model_0,
    train_dataloader,
    test_dataloader,
    # multiclass data -> Cross Entropy
    nn.CrossEntropyLoss(),
    torch.optim.SGD,
    0.1,
    3, # few epochs at first
    RND_SEED,
    # Evaluating with accuracy
    accuracy_fn,
    True
)
model_0_results

## Build a better model with non-linearity


In [ ]:
import torchmetrics as tm
from helper_functions import accuracy_fn
from timeit import default_timer as timer

class FashionV1(nn.Module):
    def __init__(self,
                 input_shape: tuple[int, int],
                 hidden_units: int,
                 output_shape: int):
        super().__init__()

        height, width = input_shape

        self.layer_stack = nn.Sequential(
            nn.Flatten(), # first format the data
            nn.Linear(
                in_features=height*width,
                out_features=hidden_units
            ),
            nn.ReLU(),
            nn.Linear(
                in_features=hidden_units,
                out_features=output_shape
            ),
            nn.ReLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layer_stack(x)

torch.manual_seed(RND_SEED)

model_1 = FashionV1(
    input_shape=(int(HEIGHT), int(WIDTH)),
    hidden_units=10,
    output_shape=len(class_names) # one for every class
).to(device)

print(model_1)
print("-----------------")

model_1_results = train_model_with_batches(
    model_1,
    train_dataloader,
    test_dataloader,
    # multiclass data -> Cross Entropy
    nn.CrossEntropyLoss(),
    torch.optim.SGD,
    0.1,
    3, # few epochs at first
    RND_SEED,
    # Evaluating with accuracy
    accuracy_fn,
    True
)
model_1_results

## Build a Convolutional Model

A CNN (Convolutional Neural Network) takes advantage of the structural disposition of the received data
That is, it can find patterns based on how the data is multidimentionaly organized in a Tensor, instead of Flattening it first

https://poloclub.github.io/cnn-explainer/


### New Layers for 2D images:
* `nn.Conv2d`: A layer of convolutional network
    > **Special Hyperparameters**:
    * `kernel_size`:
        * The dimentions of the Sliding Window, also called filter size
        * The number of image units/pixels grouped at time that this layer will try to find a pattern on the input
        * Lower values can find smaller patterns, and higher, larger
    * `stride`:
        * Number of pixels to shift between operations
        * That is, if the stride is 1, the filter will move through the image 1 pixel at time
    * `padding`:
        * If set to 'valid' or '1', padds the tensor with zeroes
        * That way, the kernel/filter slides through more values and the output shape becomes larger (as the filter will have more space to move)
* `nn.MaxPool2d`: Outputs the highest value from a tensor
    * Indicates the tensor more likeable of containing important information
    > **Special Hyperparameters**:
    * `kernel_size`:
        * with a kernel size of 2, the size in all dimentions is reduced by half
        * So, for 2D, the number of values in the tensor is reduced to 1/4
            

In [ ]:
class FashionV2(nn.Module):
    def __init__(self,
                 # Not a tuple this time, only the number of colors
                 input_shape: int,
                 n_hidden_units: int,
                 output_shape: int
                 ):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(
                in_channels=input_shape,
                out_channels=n_hidden_units,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),
            nn.Conv2d(
                in_channels=n_hidden_units,
                out_channels=n_hidden_units,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
            )
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(
                in_channels=n_hidden_units,
                out_channels=n_hidden_units,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),
            nn.Conv2d(
                in_channels=n_hidden_units,
                out_channels=n_hidden_units,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
            )
        )
        self.out_layer = nn.Sequential(
            nn.Flatten(),
            nn.Linear(
                in_features=n_hidden_units*7*7,
                out_features=output_shape,
            ),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # print("Original Shape:", x.shape)
        x = self.conv_block_1(x)
        # print("After Layer 1: ", x.shape)
        x = self.conv_block_2(x)
        # print("After Layer 2: ", x.shape)
        x = self.out_layer(x)
        # print("After Max Pool:", x.shape)
        return x

torch.manual_seed(RND_SEED)
model_2 = FashionV2(
    COLORS,
    10,
    len(class_names),
).to(device)

model_2_results = train_model_with_batches(
    model_2,
    train_dataloader,
    test_dataloader,
    # multiclass data -> Cross Entropy
    nn.CrossEntropyLoss(),
    torch.optim.SGD,
    0.1,
    3, # few epochs at first
    RND_SEED,
    # Evaluating with accuracy
    accuracy_fn,
    True
)
model_2_results

# Comparison

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    model_0_results,
    model_1_results,
    model_2_results,
])

comparison.set_index("model_name")["model_eval (%)"].plot(kind="barh")
plt.xlabel("Accuracy (%)")
plt.ylabel("model")

comparison

## Plotting Predicitions

In [ ]:
import random
def sample_data(
    data: tv.datasets.FashionMNIST,
    k: int,
    RND_SEED:int=None,
    ) -> tuple[list, list]:

    if not RND_SEED is None:
        random.seed(RND_SEED)
    samples, labels = [], []

    for sample, label in random.sample(list(data), k=k):
        samples.append(sample)
        labels.append(label)

    print()

    return samples, labels


def make_predictions(
        model: nn.Module,
        data: list[torch.Tensor],
    ) -> torch.Tensor:
    pred_probs = []

    device = next(model.parameters()).device
    model.eval()

    with torch.inference_mode():
        for sample in tqdm(data, desc="Making Predictions"):
            sample = sample.unsqueeze(dim=0).to(device)

            # Raw logits from forward pass
            pred_logits = model(sample)
            # Logits -> probability
            pred_prob = torch.softmax(pred_logits.squeeze(), dim=0)

            # must be in cpu for plotting
            pred_probs.append(pred_prob.cpu())

    # Stacks the probs to turn into a tensor
    return torch.stack(pred_probs).argmax(dim=1)

def plot_predicitions(
        imgs: list[torch.Tensor],
        preds: torch.Tensor,
        labels: list[int],
        class_names: list[str],
    ):

    SIZE = len(labels)

    plt.figure(figsize=(SIZE, SIZE))
    ROWS = int(SIZE**(0.5))
    COLS = SIZE//ROWS
    for i in range(SIZE):
        img = imgs[i].squeeze()
        test_label = class_names[preds[i]]
        right_label = class_names[labels[i]]

        plt.subplot(ROWS, COLS, i+1)
        plt.imshow(img, cmap="gray")
        if right_label == test_label:
            plt.title(test_label, c='g', backgroundcolor='black')
        else:
            plt.title(f"T: {right_label} | F: {test_label}", c='r', backgroundcolor='black')
        plt.axis(False)

In [ ]:
# make said predictions

samples, labels = sample_data(
    test_data, 9, 1,
)
preds = make_predictions(
    model_2, samples,
)
plot_predicitions(
    samples, preds, labels, class_names,
)

## Confusion Matrix

The most complete and easy to understand way to get a Classification Model Raw Metrics

Can be plotted using `mlxtend.plotting.plot_confusion_matrix()`

In [ ]:
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix

y_preds_list = []

with torch.inference_mode():
    for X, _ in tqdm(test_dataloader, desc="Predicting..."):
        y_preds_list.append(model_2(X.to(device)).squeeze().softmax(dim=0).argmax(dim=1).cpu())

y_preds = torch.cat(y_preds_list)

cfnm = ConfusionMatrix(num_classes=len(class_names), task='multiclass')
cfnm_tensor = cfnm(
    preds=y_preds,
    target=test_data.targets
)

fig, ax = plot_confusion_matrix(
    conf_mat=cfnm_tensor.numpy(),
    class_names=class_names,
    figsize=(10, 7),
)

# Saving all models

In [ ]:
from pathlib import Path

MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

for i, model in enumerate([model_0, model_1, model_2]):
    name = f"cv_{i}.pth"
    path = MODEL_PATH / name
    print("saving to", path)
    torch.save(obj=model.state_dict(), f=path)

In [ ]:
# Loading models and verifying

for i, res in enumerate([model_0_results, model_1_results, model_2_results]):
    torch.manual_seed(RND_SEED)
    new_model = [FashionV0, FashionV1, FashionV2][i](
        COLORS if i == 2 else (HEIGHT, WIDTH),
        10,
        len(class_names),
    )
    name = f"cv_{i}.pth"
    path = MODEL_PATH / name
    model_dict = torch.load(f=path)
    new_model.load_state_dict(model_dict)
    new_model.to(device)

    loaded_res = eval_model(
        new_model, test_dataloader, nn.CrossEntropyLoss(), torch.device(device), accuracy_fn, float(res["training_time (sec)"]),
    )
    print(res)
    print(loaded_res)
    print(torch.isclose(
        torch.tensor(res["model_loss"]),
        torch.tensor(loaded_res["model_loss"]),
        atol=1e-08,
    ))